In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATASET = "FD002"  # Change to FD001, FD002, FD003, or FD004

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / f"train_{DATASET}.txt"
TEST_PATH = PROJECT_ROOT / "data" / "raw" / f"test_{DATASET}.txt"
RUL_PATH = PROJECT_ROOT / "data" / "raw" / f"RUL_{DATASET}.txt"

RESULTS_PATH = PROJECT_ROOT / "results"
FIGURES_PATH = RESULTS_PATH / "figures" / DATASET
METRICS_PATH = RESULTS_PATH / "metrics"

FIGURES_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_ROOT)
print("Train file exists:", TRAIN_PATH.exists())
print("Test file exists:", TEST_PATH.exists())
print("RUL file exists:", RUL_PATH.exists())

In [ ]:
index_columns = [
    "engine_id",
    "cycle",
]

setting_columns = [
    "setting_1",
    "setting_2",
    "setting_3",
]

sensor_columns = [
    f"sensor_{number}" for number in range(1, 22)
]

column_names = index_columns + setting_columns + sensor_columns

print("Number of columns:", len(column_names))
print(column_names)

In [ ]:
train = pd.read_csv(
    TRAIN_PATH,
    sep=r"\s+",
    header=None,
    names=column_names
)

test = pd.read_csv(
    TEST_PATH,
    sep=r"\s+",
    header=None,
    names=column_names
)

print("Training shape:", train.shape)
print("Test shape:", test.shape)

print("\nTraining engines:", train["engine_id"].nunique())
print("Test engines:", test["engine_id"].nunique())

test.head()

In [ ]:
test_rul = pd.read_csv(
    RUL_PATH,
    sep=r"\s+",
    header=None,
    names=["RUL"]
)

test_rul["engine_id"] = np.arange(
    1,
    len(test_rul) + 1
)

test_rul = test_rul[
    ["engine_id", "RUL"]
]

print(test_rul.head())
print("\nNumber of RUL values:", len(test_rul))

In [ ]:
maximum_cycles = (
    train.groupby("engine_id")["cycle"]
    .max()
    .rename("maximum_cycle")
)

train = train.merge(
    maximum_cycles,
    on="engine_id",
    how="left",
    validate="many_to_one"
)

RUL_CAP = 100  # Set from per-dataset validation results in notebook 03

train["RUL_raw"] = (
    train["maximum_cycle"] - train["cycle"]
)

train["RUL_capped"] = train["RUL_raw"].clip(
    upper=RUL_CAP
)

print(train[
    ["engine_id", "cycle", "RUL_raw", "RUL_capped"]
].head())

In [ ]:
sensor_statistics = train[
    sensor_columns
].agg(
    ["mean", "std", "min", "max"]
).T

constant_sensors = sensor_statistics[
    sensor_statistics["std"] < 1e-8
].index.tolist()

usable_sensors = [
    sensor
    for sensor in sensor_columns
    if sensor not in constant_sensors
]

print("Constant sensors:")
print(constant_sensors)

print("\nUsable sensors:")
print(usable_sensors)

print("\nNumber of usable sensors:", len(usable_sensors))

In [ ]:
ROLLING_WINDOW = 5

# Ensure chronological order for each engine
train = train.sort_values(
    ["engine_id", "cycle"]
).reset_index(drop=True)

test = test.sort_values(
    ["engine_id", "cycle"]
).reset_index(drop=True)

def add_engineered_features(df, sensors):
    df = df.copy()

    for sensor in sensors:
        grouped = df.groupby("engine_id")[sensor]

        # Rolling mean
        df[f"{sensor}_rolling_mean"] = (
            grouped.transform(
                lambda x: x.rolling(
                    window=ROLLING_WINDOW,
                    min_periods=1
                ).mean()
            )
        )

        # Rolling standard deviation
        df[f"{sensor}_rolling_std"] = (
            grouped.transform(
                lambda x: x.rolling(
                    window=ROLLING_WINDOW,
                    min_periods=2
                ).std()
            )
        )

        # Difference between consecutive cycles
        df[f"{sensor}_delta"] = (
            grouped.diff().fillna(0)
        )

    return df


train = add_engineered_features(
    train, usable_sensors
)

test = add_engineered_features(
    test, usable_sensors
)

print("Training shape:", train.shape)
print("Test shape:", test.shape)

In [ ]:
baseline_features = (
    ["cycle"]
    + setting_columns
    + usable_sensors
)

rolling_mean_features = [
    f"{sensor}_rolling_mean"
    for sensor in usable_sensors
]

rolling_std_features = [
    f"{sensor}_rolling_std"
    for sensor in usable_sensors
]

delta_features = [
    f"{sensor}_delta"
    for sensor in usable_sensors
]

enhanced_features = (
    baseline_features
    + rolling_mean_features
    + rolling_std_features
    + delta_features
)

feature_columns = enhanced_features

X_train = train[feature_columns].copy()

y_train = train["RUL_capped"].copy()

print("Number of features:", len(feature_columns))
print("Training shape:", X_train.shape)

In [ ]:
test_final = (
    test.sort_values(
        ["engine_id", "cycle"]
    )
    .groupby(
        "engine_id",
        as_index=False
    )
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)

print("Rows in final test set:", len(test_final))

test_final[
    ["engine_id", "cycle"]
].head(10)

In [ ]:
test_final = test_final.merge(
    test_rul,
    on="engine_id",
    how="left",
    validate="one_to_one"
)

print(
    test_final[
        ["engine_id", "cycle", "RUL"]
    ].head(10)
)

assert test_final["RUL"].isna().sum() == 0
assert len(test_final) == 100

In [ ]:
improved_rf = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=4,
                max_features="sqrt",
                n_jobs=-1,
                random_state=RANDOM_STATE
            )
        )
    ]
)

# Train improved model on all FD001 training engines
improved_rf.fit(
    X_train,
    y_train
)

print("Improved Random Forest trained successfully.")

In [ ]:
X_test = test_final[
    feature_columns
].copy()

# Use actual, uncapped RUL values
y_test = test_final[
    "RUL"
].copy()

# Make predictions using improved model
test_predictions = improved_rf.predict(
    X_test
)

print("Test engines:", len(X_test))
print("Predictions:", len(test_predictions))

print("\nFirst 10 predictions:")
print(test_predictions[:10])

In [ ]:
mae = mean_absolute_error(
    y_test, test_predictions
)

rmse = np.sqrt(
    mean_squared_error(y_test, test_predictions)
)

r_squared = r2_score(
    y_test, test_predictions
)

test_metrics = pd.DataFrame({
    "Model": ["Improved Random Forest (RUL cap 100)"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r_squared]
})

print(test_metrics.round(3))

In [ ]:
predictions = pd.DataFrame(
    {
        "engine_id": test_final["engine_id"],
        "final_cycle": test_final["cycle"],
        "actual_RUL": y_test,
        "predicted_RUL": test_predictions,
    }
)

predictions["error"] = (
    predictions["predicted_RUL"]
    - predictions["actual_RUL"]
)

predictions["absolute_error"] = (
    predictions["error"].abs()
)

predictions.head(10).round(2)

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    y_test,
    test_predictions,
    alpha=0.7,
    s=40
)

minimum_value = min(
    y_test.min(),
    test_predictions.min()
)

maximum_value = max(
    y_test.max(),
    test_predictions.max()
)

plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    linestyle="--",
    label="Perfect prediction"
)

plt.xlabel("Actual RUL (cycles)")
plt.ylabel("Predicted RUL (cycles)")
plt.title(
    f"Improved {DATASET} Test Set: Actual vs Predicted RUL"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURES_PATH / "improved_test_actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.scatter(
    predictions["actual_RUL"],
    predictions["error"],
    alpha=0.7,
    s=40
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Actual RUL (cycles)")
plt.ylabel("Prediction Error (cycles)")
plt.title(
    f" Improved {DATASET} Test Set Prediction Errors"
)

plt.tight_layout()

plt.savefig(
    FIGURES_PATH / "improved_test_residuals.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
worst_predictions = (
    predictions
    .sort_values(
        "absolute_error",
        ascending=False
    )
    .head(10)
)

worst_predictions.round(2)

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    predictions["error"],
    bins=20,
    edgecolor="black"
)

plt.axvline(
    0,
    linestyle="--"
)

plt.xlabel("Prediction Error (cycles)")
plt.ylabel("Number of Engines")
plt.title(
    f"Improved Distribution of {DATASET} Test Prediction Errors"
)

plt.tight_layout()

plt.savefig(
    FIGURES_PATH / "improved_test_error_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
mean_error = predictions[
    "error"
].mean()

median_error = predictions[
    "error"
].median()

print(
    f"Mean prediction error: {mean_error:.2f} cycles"
)

print(
    f"Median prediction error: {median_error:.2f} cycles"
)

if mean_error > 0:
    print(
        "The model tends to overestimate RUL."
    )
elif mean_error < 0:
    print(
        "The model tends to underestimate RUL."
    )
else:
    print(
        "The model shows no average prediction bias."
    )

In [ ]:
test_metrics.to_csv(
    METRICS_PATH / f"{DATASET.lower()}_improved_test_metrics.csv",
    index=False
)

predictions.to_csv(
    METRICS_PATH / f"{DATASET.lower()}_improved_test_predictions.csv",
    index=False
)

print("Improved model results saved.")

In [ ]:
baseline_metrics = pd.read_csv(
    METRICS_PATH / f"{DATASET.lower()}_test_metrics.csv"
)

improved_metrics = pd.read_csv(
    METRICS_PATH / f"{DATASET.lower()}_improved_test_metrics.csv"
)

model_comparison = pd.concat(
    [baseline_metrics, improved_metrics],
    ignore_index=True
)

display(model_comparison.round(3))

model_comparison.to_csv(
    METRICS_PATH / f"{DATASET.lower()}_model_comparison.csv",
    index=False
)